# 05 - Error Analysis (hard-negative + EM fix)

**Latar belakang:** Nb04 telah diupdate:
- Negative labels jadi **hard-negative** (PRD 7) — pasangan beda customer_id yang berbagi >=1 field identitas, bukan random penuh.
- Argumen max_pairs pada EM dihapus (bukan parameter estimate_parameters_using_expectation_maximisation).
- Distribusi probability berubah drastis (mean 0.108 -> 0.425).

Notebook ini menganalisis distribusi BARU + error terhadap **manual review** dari review_queue.csv.

**Yang TIDAK dianalisis ulang:** bukan lagi soal REVIEW selalu 0 / blocking telalu sempit — itu sudah teratasi di Nb04.


## 1. Setup


In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PRED = r"C:\Users\User\Downloads\Fix\data\raw\splink_predictions_relaxed.csv"
REVIEW = r"C:\Users\User\Downloads\Fix\data\raw\review_queue.csv"
print("Setup OK")


## 2. Load predictions + manual review labels

Join by customer_id_l/r (key review_queue). Manual label = hasil review manusia (bukan silver-standard).


In [ ]:
pred = pd.read_csv(PRED)
rq = pd.read_csv(REVIEW, sep=";")
g = pred.merge(rq[["customer_id_l", "customer_id_r", "manual_label"]],
              on=["customer_id_l", "customer_id_r"], how="left")
print("Preds:", len(pred), "| direview:", g["manual_label"].notna().sum())
print(g["manual_label"].value_counts(dropna=False).to_string())


## 3. Distribusi probability (state baru)

Perhatian: mean naik 0.108 -> 0.425. REVIEW (0.20-0.80) kini terisi. Tapi cek ekor tinggi: berapa yang non-silver di >=0.8?


In [ ]:
bins = [0, 0.10, 0.20, 0.50, 0.80, 0.95, 0.999, 1.001]
lab = ["[0,0.1)", "[0.1,0.2)", "[0.2,0.5)", "[0.5,0.8)", "[0.8,0.95)", "[0.95,0.999)", "=1.0"]
cuts = pd.cut(pred["match_probability"], bins=bins, labels=lab, right=False)
print(cuts.value_counts().sort_index().to_string())

print(f"\nSilver positive @1.0: {(pred['match_probability']==1.0).sum()}")
print(f"Mean  : {pred['match_probability'].mean():.4f}")
print(f"Median: {pred['match_probability'].median():.4f}")


## 4. Error terhadap manual review

Manual labels stratified (B = low-prob studi diambil banyak), jadi precision di sini BUKAN yang global — hanya indikasi.


In [ ]:
g2 = g[g["manual_label"].isin(["match", "non-match"])].copy()
g2["y"] = (g2["manual_label"] == "match").astype(int)

bins2 = [0, 0.10, 0.20, 0.50, 0.80, 0.95, 1.001]
lab2 = ["[0,0.1)", "[0.1,0.2)", "[0.2,0.5)", "[0.5,0.8)", "[0.8,0.95)", "[0.95,1.0]"]
cuts2 = pd.cut(g2["match_probability"], bins=bins2, labels=lab2, right=False)
print(pd.crosstab(cuts2, g2["manual_label"], margins=True).to_string())

print("\nThreshold sweep (dari manual labels):")
for t in [0.20, 0.50, 0.80, 0.95]:
    yp = (g2["match_probability"] >= t).astype(int)
    tp = int(((g2["y"] == 1) & (yp == 1)).sum())
    fp = int(((g2["y"] == 0) & (yp == 1)).sum())
    fn = int(((g2["y"] == 1) & (yp == 0)).sum())
    tn = int(((g2["y"] == 0) & (yp == 0)).sum())
    pr = tp / (tp + fp) if tp + fp else 0
    rc = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0
    print(f"  thr {t:.2f}: TP={tp:5d} FP={fp:5d} FN={fn:3d} TN={tn:5d}  P={pr:.3f} R={rc:.3f} F1={f1:.3f}")


## 5. FP di ekor tinggi (prob >= 0.95)

Manual non-match dengan prob >=0.95 = model yakin tapi salah. Karakterisasi pola field-nya.


In [ ]:
fp = g[(g["manual_label"] == "non-match") & (g["match_probability"] >= 0.95)]
print(f"FP manual prob>=0.95: {len(fp)}\n")
print("gamma_email:", fp["gamma_email_std"].value_counts().to_dict())
print("gamma_first:", fp["gamma_first_name_std"].value_counts().to_dict())
print("gamma_last :", fp["gamma_last_name_std"].value_counts().to_dict())
print("gamma_dob  :", fp["gamma_dob_std"].value_counts().to_dict())
print("match_key  :", fp["match_key"].value_counts().to_dict())


Inspeksi beberapa pair FP untuk memahami mengapa prob tinggi.


In [ ]:
cols = ["match_probability", "email_std_l", "email_std_r",
        "gamma_email_std", "first_name_std_l", "last_name_std_l",
        "first_name_std_r", "last_name_std_r", "gamma_dob_std"]
if len(fp):
    print(fp[cols].head(10).to_string())
else:
    print("Tidak ada FP prob>=0.95.")


## 6. FN: manual match dengan prob rendah


In [ ]:
fn = g[(g["manual_label"] == "match") & (g["match_probability"] < 0.80)]
print(f"Manual MATCH tapi prob<0.80: {len(fn)}")
if len(fn):
    print(fn[["match_probability", "match_weight", "match_key"]].describe().round(3).to_string())
else:
    print("Tidak ada FN — semua manual match di >=0.80.")


## 7. Root cause: parameter m/u belum terlatih

Warnings Nb04 mengatakan m/u email, phone, dob, address, city belum terlatih penuh. Level fuzzy (lev<=1, lev<=2, jaro, date diff) menggunakan default Splink. Ini alasan FP ekor tetap tinggi: pasangan nama-exact + email-same-user beda-domain mendapat bobot email dari m default (belum dilatih pada negative email-serupa).


In [ ]:
print("Parameter yang BELUM terlatih (dari output Nb04):")
print("  - email_std   : m, u")
print("  - phone_main  : m")
print("  - dob_std     : m, u")
print("  - address_std : m")
print("  - city_std    : m")
print()
print("Mengapa level ini tak teramati?")
print("  Semua silver positive (1.867) = duplikat TRIVIALLY IDENTICAL.")
print("  email+phone+dob = EXACT sama. Jadi level fuzzy phone/dob/email")
print("  TIDAK PERNAH muncul pada true match -> m fuzzy tak bisa dilatih -> default.")
print()
print("FP 0.95 (nama+email-same-user, beda domain, beda dob):")
print("  email gamma 2 (Jaro-Winkler >= 0.88) -> m email belum terlatih -> default = tinggi")
print("  dob gamma -1  (beda) -> m dob belum terlatih -> default cukup rendah")
print("  Kombinasi -> prob tinggi -> FP. Solusi: lengkapi m utk level fuzzy email/dob.")


## 8. Kesimpulan & rekomendasi

### Yang sudah membaik
- Saturasi global hilang: mean 0.425, distribusi menyebar.
- REVIEW 0.20-0.80 kini terisi (bukan lagi 0 pair).
- 1.867 silver positive semua prob=1.0 dan benar same-customer.
- Hard-negative training menjadikan negative tidak lagi random penuh — lebih realistis.

### Yang tersisa (bukan salah threshold)
- Ekor 0.8-1.0 masih berisi non-silver: ~1.493 pair, di antaranya **226 manual non-match di >=0.95**.
- Pola dominan FP: nama exact + email same-user beda domain + dob beda.
- Root cause: m/u level fuzzy (email, dob, address, city) BELUM terlatih -> default.

### Rekomendasi selanjutnya (satu komponen saja)
1. Lengkapi m/u fuzzy: tambah hard-negative yang mencakup email-username-same beda-domain ke reference pair Nb04 step 3.
2. Ukur ulang: harapkan 226 FP turun signifikan, distribusi ekor kanan bergeser ke 0.20-0.80.
3. Setelah distribusi stabil, baru tuning threshold (Nb06/08).

JANGAN langsung ubah threshold 0.85/0.20 sebelum m/u fuzzy lengkap — itu menghias gejala, bukan akar.


---

*Nb05 selesai. Lanjut ke Nb06 setelah m/u fuzzy dilengkapi di Nb04.*
